# Generate QA Dataset for an Existing Corpus

Creates (or recreates) a `train_questions.parquet` for an **already-indexed**
corpus so it can be evaluated with `rag_evaluation.ipynb`.

**Workflow:**
1. Set `NAME` to the target collection (must already have `wiki_corpus.parquet`)
2. Choose QA sources & balancing options
3. Run all cells — loads, enriches, balances, saves
4. Open `rag_evaluation.ipynb` with the same `NAME` and evaluate

In [ ]:
from pathlib import Path
import pandas as pd
from config import DATA_DIR, CACHE_DIR

# ── Target collection (must already have wiki_corpus.parquet) ────────────────
NAME = "wiki_full"
COLLECTION_ROOT = Path(DATA_DIR) / NAME
WIKI_PARQUET   = COLLECTION_ROOT / "wiki_corpus.parquet"
QUESTIONS_PATH = COLLECTION_ROOT / "test_2.parquet"
    
# ── QA sources (HuggingFace config names) ────────────────────────────────────
QA_DATASETS = ['natural_questions']
POPULARITY_DATASET = "Cyro1/enwiki_pageviews_m"


# ── Decile balancing ──────────────────────────────────────────────────────────
BALANCE = True
BALANCE_DATASETS = True
TARGET_PER_DECILE = 30                      

# ── Synthetic generation (optional) ─────────────────────────────────────────
GENERATE_SYNTHETIC = False
QUESTIONS_PER_DECILE = 300
MODEL_NAME = "gpt-4.1-nano"

# ── Decile calculation (chunk-based weighting) ───────────────────────────────
WEIGHT_BY_CHUNKS = True
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# ── Sanity check ─────────────────────────────────────────────────────────────
assert WIKI_PARQUET.exists(), f"Corpus not found: {WIKI_PARQUET}"
print(f"✓ Collection: {NAME}")
print(f"  Corpus:     {WIKI_PARQUET}  ({WIKI_PARQUET.stat().st_size / 1e9:.2f} GB)")
print(f"  QA sources: {QA_DATASETS}")
print(f"  Dataset balance: {BALANCE_DATASETS}  |  Decile balance: {BALANCE}  |  Synthetic: {GENERATE_SYNTHETIC}")
print(f"  Deciles:    {'chunk-weighted' if WEIGHT_BY_CHUNKS else 'unweighted'} (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP})")


✓ Collection: wiki_full
  Corpus:     /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/wiki_corpus.parquet  (9.88 GB)
  QA sources: ['natural_questions', 'fever', 'hotpot_qa', 'trex', 'pop_qa', 'trivia_qa']
  Dataset balance: True  |  Decile balance: True  |  Synthetic: False
  Deciles:    chunk-weighted (chunk_size=1000, chunk_overlap=100)


In [2]:
from scripts.prepare_qa_dataset import prepare_qa_dataset

qa_df = prepare_qa_dataset(
    qa_datasets=QA_DATASETS,
    popularity_dataset=POPULARITY_DATASET,
    output_path=QUESTIONS_PATH,
    balance_datasets=BALANCE_DATASETS,
    balance=BALANCE,
    target_per_decile=TARGET_PER_DECILE,
    generate_synthetic=GENERATE_SYNTHETIC,
    corpus_path=WIKI_PARQUET,
    questions_per_decile=QUESTIONS_PER_DECILE,
    model_name=MODEL_NAME,
    cache_dir=CACHE_DIR,
    weight_by_chunks=WEIGHT_BY_CHUNKS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    metadata_path=COLLECTION_ROOT / "metadata.json",
)


INFO - PREPARE QA DATASET
INFO - [1/4] LOAD
/Users/cyro/Documents/VSC/PopularityBias/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
INFO - PyTorch version 2.8.0 available.
INFO - Loading natural_questions...
INFO -   ✓ 81,533 questions
INFO - Loading fever...
INFO -   ✓ 94,367 questions
INFO - Loading hotpot_qa...
INFO -   ✓ 148,414 questions
INFO - Loading trex...
INFO -   ✓ 2,839,105 questions
INFO - Loading pop_qa...
INFO -   ✓ 13,811 questions
INFO - Loading trivia_qa...
INFO -   ✓ 98,386 questions
INFO - Total: 3,275,616 questions
INFO - [2/4] FILTER TO CORPUS
INFO - Loading corpus IDs...
INFO - Kept 3,275,616 / 3,275,616 questions (in corpus)
INFO - [3/4] ASSIGN DECILES
INFO - Calculating deciles from corpus...
INFO - Using chunk-weighted deciles (chunk_size=1000, chunk_overlap=100)

Corpus boundaries:   0%|          | 0/60 [00:00<?, ?it/s]

INFO - Unique docs with popularity: 5,890,044  |  total chunks: 24,651,978
INFO - QA decile distribution (chunk-weighted): {0: 746688, 1: 524998, 2: 411568, 3: 335482, 4: 270406, 5: 219182, 6: 181199, 7: 153082, 8: 144426, 9: 288585}
INFO - Total corpus docs: 5,903,530 | Unique: 5,890,044
INFO - Saved collection metadata to /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/metadata.json
INFO - [4.5/5] BALANCE PER DATASET
INFO -   Per-dataset [pop_qa]: 13,811 available → 50 taken
INFO -   Per-dataset [natural_questions]: 81,533 available → 50 taken
INFO -   Per-dataset [fever]: 94,367 available → 50 taken
INFO -   Per-dataset [trivia_qa]: 98,386 available → 50 taken
INFO -   Per-dataset [hotpot_qa]: 148,414 available → 50 taken
INFO -   Per-dataset [trex]: 2,839,105 available → 50 taken
INFO - After per-dataset balance: 300 questions total
INFO - [4/4] BALANCE
INFO - Balancing to 30 per decile...
WARNING - Decile 0: 14 (short by 16)
WARNING - Decile 1: 17 (short by 13)
WARNING - D

In [3]:

# ── Check corpus decile distribution ──────────────────────────────────────────
import pyarrow.parquet as pq
from helpers.decile_utils import COL_DECILE_UNWEIGHTED, COL_DECILE_CHUNK_WEIGHTED

print("Checking corpus decile distribution...")
parquet_file = pq.ParquetFile(WIKI_PARQUET)
available_cols = parquet_file.schema_arrow.names
print(f"Corpus columns: {available_cols}")

# Determine which decile column to inspect (prefer chunk-weighted, fall back to unweighted, then legacy 'decile')
DECILE_COL = None
for candidate in [COL_DECILE_CHUNK_WEIGHTED, COL_DECILE_UNWEIGHTED, "decile"]:
    if candidate in available_cols:
        DECILE_COL = candidate
        break

if DECILE_COL is None:
    if "popularity_avg" in available_cols:
        print("\nℹ️  No pre-computed decile column found in corpus.")
        print("   The corpus contains 'popularity_avg' — deciles will be computed")
        print("   on-the-fly by prepare_qa_dataset using the corpus boundaries.")
    else:
        print("\n⚠️  WARNING: No decile or popularity column found in the corpus!")
        print("   Ensure the corpus was built with popularity data.")
else:
    # Sample deciles from corpus in chunks
    decile_counts = {}
    sample_size = 0
    for i, batch in enumerate(parquet_file.iter_batches(batch_size=100_000, columns=[DECILE_COL])):
        batch_df = batch.to_pandas()
        for decile, count in batch_df[DECILE_COL].value_counts().items():
            decile_counts[decile] = decile_counts.get(decile, 0) + count
        sample_size += len(batch_df)
        del batch_df
        if i >= 5:  # Check first ~500k docs
            break

    print(f"\nCorpus '{DECILE_COL}' distribution (first {sample_size:,} docs):")
    for decile in sorted(decile_counts.keys()):
        print(f"  Decile {decile}: {decile_counts[decile]:,}")

    if len(decile_counts) == 1 and 0 in decile_counts:
        print("\n⚠️  WARNING: Corpus has ALL documents in decile 0!")
        print("   This is incorrect. The corpus needs proper deciles assigned.")
        print("   You need to regenerate the corpus with correct deciles from popularity data.")


Checking corpus decile distribution...
Corpus columns: ['wikipedia_id', 'wikipedia_title', 'text', 'popularity_avg', 'popularity_rank', 'decile']

Corpus 'decile' distribution (first 600,000 docs):
  Decile -1: 1,334
  Decile 0: 60,329
  Decile 1: 58,896
  Decile 2: 58,196
  Decile 3: 60,133
  Decile 4: 59,382
  Decile 5: 60,133
  Decile 6: 59,728
  Decile 7: 59,962
  Decile 8: 60,102
  Decile 9: 61,805


In [4]:
# ── Quick inspection ──────────────────────────────────────────────────────────
print(f"Saved: {QUESTIONS_PATH}")
print(f"Total: {len(qa_df):,} questions\n")

if "decile" in qa_df.columns:
    print("Per-decile distribution:")
    print(qa_df["decile"].value_counts().sort_index())

if "dataset" in qa_df.columns:
    print(f"\nSources:")
    print(qa_df["dataset"].value_counts())

display(qa_df.sample(5, random_state=42))

Saved: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/test_1.parquet
Total: 197 questions

Per-decile distribution:
decile
0    14
1    17
2    10
3    12
4    14
5    21
6    23
7    28
8    28
9    30
Name: count, dtype: int64

Sources:
dataset
trex                 49
pop_qa               44
hotpot_qa            36
natural_questions    35
fever                17
trivia_qa            16
Name: count, dtype: int64


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank,dataset,pop_decile_unweighted,pop_decile_chunk_weighted,decile
140,1711457,What is the religion of Philip Neri?,[Catholic Church],304546,Philip Neri,6098.562500,2.048116e+05,pop_qa,9,8,8
113,4663230,In what country is Burns?,[United States of America],130781,"Burns, Oregon",2745.416667,3.561474e+05,pop_qa,9,7,7
16,3101008,In what city was Anatoly Georgievich Ufimtsev ...,[Kursk],24141083,Anatoly Georgievich Ufimtsev,26.000000,4.060288e+06,pop_qa,3,1,1
75,1506231216119364910,who is jerome avenue in the bronx named after,[Leonard Jerome],14110643,Jerome Avenue,686.729167,9.157550e+05,natural_questions,8,5,5
155,213100,Marlon Brando was in a film.,"[SUPPORTS, SUPPORTS, SUPPORTS, SUPPORTS, SUPPO...",2268630,The Missouri Breaks,7753.687500,1.616256e+05,fever,9,8,8


In [5]:
# ── Verify overlap with corpus ────────────────────────────────────────────────
corpus_ids = set(pd.read_parquet(WIKI_PARQUET, columns=["wikipedia_id"])["wikipedia_id"].astype(int))
qa_ids     = set(qa_df["wikipedia_id"].astype(int))

in_corpus = qa_ids & corpus_ids
missing   = qa_ids - corpus_ids

print(f"QA doc IDs in corpus: {len(in_corpus):,} / {len(qa_ids):,}  ({100 * len(in_corpus) / len(qa_ids):.1f}%)")
if missing:
    print(f"⚠️  {len(missing):,} QA doc IDs NOT in corpus — these questions can never be answered correctly")
else:
    print("✓ All QA documents exist in the corpus")

QA doc IDs in corpus: 197 / 197  (100.0%)
✓ All QA documents exist in the corpus
